In [1]:
from pathlib import Path
import json

pw_file = Path.cwd().parent.parent / ".json"
with open(pw_file, encoding="utf-8") as f:
    my_file = json.load(f)

PORTAL_ID = my_file["PORTAL_ID"]
USERNAME = my_file["USER"]
PASSWORD = my_file["VR_PW"]

BROKER = "mqtt.victronenergy.com"
PORT = 8883
SUB_TOPIC = f"N/{PORTAL_ID}/#"
KEEPALIVE_TOPIC = f"R/{PORTAL_ID}/keepalive"


In [2]:
import paho.mqtt.client as mqtt
import ssl

def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("Verbunden!")

        # Erst abonnieren
        client.subscribe(SUB_TOPIC)
        print(f"Abonniert: {SUB_TOPIC}")

        # Dann Keepalive senden (triggert Datenfluss)
        payload = json.dumps({"keepalive-options": ["full"]})
        client.publish(KEEPALIVE_TOPIC, payload)
        print(f"Keepalive gesendet an {KEEPALIVE_TOPIC}")

    else:
        print("Verbindung fehlgeschlagen, Code:", rc)


def on_message(client, userdata, msg):
    try:
        print(f"{msg.topic} -> {msg.payload.decode()}")
    except:
        print(f"{msg.topic} -> (binary payload)")


client = mqtt.Client(
    client_id="milp_client",
    protocol=mqtt.MQTTv311,
)

client.username_pw_set(USERNAME, PASSWORD)

client.tls_set(
    ca_certs="../data/venus-ca.crt",  # Victron CA Datei
    cert_reqs=ssl.CERT_REQUIRED
)
client.tls_insecure_set(False)

client.on_connect = on_connect
client.on_message = on_message

print("Verbinde...")
client.connect(BROKER, PORT, keepalive=30)

client.loop_forever()


Verbinde...


/tmp/ipykernel_2658007/1960316977.py:28: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(


Verbunden!
Abonniert: N/c0619ab9ba54/#
Keepalive gesendet an R/c0619ab9ba54/keepalive
N/c0619ab9ba54/system/0/Serial -> {"value":"c0619ab9ba54"}
N/c0619ab9ba54/pvinverter/0/Devices/0/ProductName -> {"value":"Energy Meter VM-3P75CT"}
N/c0619ab9ba54/pvinverter/0/IsGenericEnergyMeter -> {"value":1}
N/c0619ab9ba54/pvinverter/0/FirmwareVersion -> {"value":67327}
N/c0619ab9ba54/pvinverter/0/Ac/L1/Energy/Forward -> {"value":0.38999998569488525}
N/c0619ab9ba54/pvinverter/0/Devices/0/CustomName -> {"value":"PV Anlage"}
N/c0619ab9ba54/pvinverter/0/ErrorCode -> {"value":0}
N/c0619ab9ba54/pvinverter/0/Ac/L1/VoltageLineToLine -> {"value":null}
N/c0619ab9ba54/pvinverter/0/Ac/L1/Energy/Reverse -> {"value":0.019999999552965164}
N/c0619ab9ba54/pvinverter/0/Mgmt/ProcessVersion -> {"value":"3.56"}
N/c0619ab9ba54/pvinverter/0/Devices/0/DeviceInstance -> {"value":0}
N/c0619ab9ba54/pvinverter/0/Ac/L1/Power -> {"value":5.09375}
N/c0619ab9ba54/pvinverter/0/Ac/L1/Current -> {"value":0.05999999865889549}
N/c061

KeyboardInterrupt: 